In [4]:
import boto3

comprehend = boto3.client('comprehend', region_name='us-east-1')

test_text = "They never responded to my dispute and ruined my credit score."
response = comprehend.detect_sentiment(Text=test_text, LanguageCode='en')

print(response['Sentiment'])
print(response['SentimentScore'])

NEGATIVE
{'Positive': 0.0022470299154520035, 'Negative': 0.992938756942749, 'Neutral': 0.002078952034935355, 'Mixed': 0.0027351723983883858}


In [5]:
import pandas as pd
import boto3

s3 = boto3.client('s3', region_name='us-east-1')

bucket = 'pavan-ravuri'
key = 'raw/cfpb_complaints_sample.csv'

obj = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(obj['Body'])

print(df.shape)
print(df.columns.tolist())
df.head(3)

(5000, 10)
['complaint_id', 'date_received', 'product', 'sub_product', 'issue', 'company', 'state', 'narrative', 'company_response', 'timely_response']


,complaint_id,date_received,product,sub_product,issue,company,state,narrative,company_response,timely_response
0,22840280,2026-06-03T09:16:12.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,"EQUIFAX, INC.",GA,This CFPB complaint has been filed to request ...,Closed with explanation,Yes
1,22840276,2026-06-03T09:13:26.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",GA,This CFPB complaint has been filed to request ...,Closed with explanation,Yes
2,22840633,2026-06-03T09:43:21.000Z,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,"EQUIFAX, INC.",OH,I am submitting this formal complaint regardin...,Closed with non-monetary relief,Yes


In [7]:
def truncate_bytes(text, max_bytes=4900):
    encoded = text.encode('utf-8')
    if len(encoded) <= max_bytes:
        return text
    return encoded[:max_bytes].decode('utf-8', errors='ignore')

sample = df['narrative'].dropna().head(20).apply(truncate_bytes).tolist()

response = comprehend.batch_detect_sentiment(
    TextList=sample,
    LanguageCode='en'
)

for r in response['ResultList']:
    idx = r['Index']
    print(sample[idx][:60], '...', '->', r['Sentiment'])

This CFPB complaint has been filed to request pursuant to FC ... -> NEGATIVE
This CFPB complaint has been filed to request pursuant to FC ... -> NEGATIVE
I am submitting this formal complaint regarding multiple XXX ... -> NEGATIVE
I am submitting this formal complaint regarding multiple XXX ... -> NEGATIVE
This is an improper framework to be dismissed and exploited  ... -> NEGATIVE
Hello, I am writing to delete the following information on m ... -> NEUTRAL
Hello, I am writing to delete the following information on m ... -> NEUTRAL
I am filing this complaint because certain information appea ... -> NEUTRAL
I was completely unaware of these accounts and did not provi ... -> NEUTRAL
Can you verify this entry? It's delayed and potentially wron ... -> NEGATIVE
Can you verify this entry? It's delayed and potentially wron ... -> NEGATIVE
I am writing to formally dispute the following information i ... -> NEUTRAL
This is my fourth ( 4th ) formal notice regarding your ongoi ... -> NEUTRAL
You h

In [8]:
import time

N = 500
texts = df['narrative'].dropna().head(N).apply(truncate_bytes).tolist()
ids = df['complaint_id'].loc[df['narrative'].dropna().head(N).index].tolist()

results = []
batch_size = 25

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    resp = comprehend.batch_detect_sentiment(TextList=batch, LanguageCode='en')
    for r in resp['ResultList']:
        idx = r['Index']
        results.append({
            'complaint_id': ids[i + idx],
            'sentiment': r['Sentiment'],
            'positive_score': r['SentimentScore']['Positive'],
            'negative_score': r['SentimentScore']['Negative'],
            'neutral_score': r['SentimentScore']['Neutral'],
            'mixed_score': r['SentimentScore']['Mixed'],
        })
    time.sleep(0.2)

sentiment_df = pd.DataFrame(results)
print(sentiment_df.shape)
sentiment_df.head()

(500, 6)


,complaint_id,sentiment,positive_score,negative_score,neutral_score,mixed_score
0,22840280,NEGATIVE,0.003334,0.793048,0.202085,0.001533
1,22840276,NEGATIVE,0.003334,0.793048,0.202085,0.001533
2,22840633,NEGATIVE,0.004025,0.853295,0.113263,0.029416
3,22840634,NEGATIVE,0.004025,0.853295,0.113263,0.029416
4,22840666,NEGATIVE,0.000441,0.949864,0.048056,0.001640


In [9]:
csv_buffer = sentiment_df.to_csv(index=False)

s3.put_object(
    Bucket='ravuri-kumar',
    Key='sentiment/cfpb_sentiment_results.csv',
    Body=csv_buffer
)

print("Saved to s3://ravuri-kumar/sentiment/cfpb_sentiment_results.csv")

Saved to s3://ravuri-kumar/sentiment/cfpb_sentiment_results.csv
